In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
DATA_DIR = Path("src")

SOURCE_FILES = {
    2022: DATA_DIR / "bund-2022.xlsx",
    2023: DATA_DIR / "bund-2023.xlsx",
    2024: DATA_DIR / "bund-2024.xlsx",
}

SHEETS = {
    "by_article": "csv-24311-05",
    "decision_type": "csv-24311-07",
    "sanction_type": "csv-24311-09",
    "citizenship": "csv-24311-47",
    "year_totals": "csv-24311-03",
}

OUTPUT_FILES = {
    "by_article": DATA_DIR / "by_article.csv",
    "decision_type": DATA_DIR / "decision_type.csv",
    "sanction_type": DATA_DIR / "sanction_type.csv",
    "citizenship": DATA_DIR / "citizenship.csv",
    "year_totals": DATA_DIR / "year_totals.csv",
}

for year, path in SOURCE_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing source file for {year}: {path}")

In [ ]:
def load_sheet_for_all_years(sheet_name: str) -> pd.DataFrame:
    frames = []
    for year, path in SOURCE_FILES.items():
        frame = pd.read_excel(path, sheet_name=sheet_name)
        frame["source_year"] = year
        frame["source_file"] = path.name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


df_by_article = load_sheet_for_all_years(SHEETS["by_article"])
df_decision_type = load_sheet_for_all_years(SHEETS["decision_type"])
df_sanction_type = load_sheet_for_all_years(SHEETS["sanction_type"])
df_citizenship = load_sheet_for_all_years(SHEETS["citizenship"])
df_year_totals = load_sheet_for_all_years(SHEETS["year_totals"])

df_year_totals = df_year_totals.assign(
    Gebiet=df_year_totals["Gebiet"].astype(str).str.strip(),
    Straftat=df_year_totals["Straftat"].astype(str).str.strip(),
    Personengruppe=df_year_totals["Personengruppe"].astype(str).str.strip(),
    Angewandtes_Strafrecht=df_year_totals["Angewandtes_Strafrecht"].astype(str).str.strip(),
    Geschlecht=df_year_totals["Geschlecht"].astype(str).str.strip(),
    Staatsangehoerigkeit=df_year_totals["Staatsangehoerigkeit"].astype(str).str.strip(),
)

df_year_totals = df_year_totals.loc[
    (df_year_totals["Gebiet"] == "Deutschland")
    & (df_year_totals["Straftat"] == "Alle Straftaten zusammen")
    & (df_year_totals["Geschlecht"] == "Insgesamt")
    & (df_year_totals["Staatsangehoerigkeit"] == "Insgesamt")
    & (df_year_totals["Angewandtes_Strafrecht"] == "Allgemeines Strafrecht und Jugendstrafrecht")
    & (df_year_totals["Personengruppe"].isin(["Abgeurteilte insgesamt", "Verurteilte insgesamt"]))
].copy()

datasets = {
    "by_article": df_by_article,
    "decision_type": df_decision_type,
    "sanction_type": df_sanction_type,
    "citizenship": df_citizenship,
    "year_totals": df_year_totals,
}

In [ ]:
for name, frame in datasets.items():
    print(f"{name}: {frame.shape}")

df_by_article.head()

In [ ]:
for name, frame in datasets.items():
    output_path = OUTPUT_FILES[name]
    frame.to_csv(output_path, index=False)
    print(f"Wrote {output_path} ({len(frame)} rows)")